# B06A — Audit adaptasi feature-distribution KD untuk klasifikasi
**Status: rancangan yang terinspirasi distillation SSID-KD, bukan reproduksi SSID-KD asli.**

Repositori asli melakukan deraining dan konsistensi representasi domain sintetis–nyata. Di sini kita mengubahnya menjadi transfer distribusi fitur Teacher beku → Student pada citra mammografi yang sama. Tidak ada klaim bahwa feature-KL ini unik untuk SSID-KD atau setara dengan metode asli.

Sumber yang diperiksa: [README](https://github.com/cuiyixin555/SSID-KD) dan [training asli](https://github.com/cuiyixin555/SSID-KD/blob/main/train_Rain200H_real.py). Perubahan eksplisit: output restorasi → logit klasifikasi; rekonstruksi/dark-channel/TV dihapus; representasi dua domain → GAP Teacher–Student; ditambah projector Student; KL dinormalisasi sebagai jumlah kanal kemudian rata-rata batch, bukan menyalin faktor 1e-6/reduction asli.

**Jalankan Cell 1–6 pada Colab GPU baru.** Default CBIS-DDSM. Setelah selesai unduh ZIP; ulangi untuk CMMD, INbreast, MIAS dengan mengganti DATASET. Tidak menjalankan eksperimen penuh atau evaluasi test. B04/B05 tidak ditulis ulang. Satu langkah diagnostik hanya terjadi pada salinan model di RAM; bobot tersebut dibuang.

Audit memakai subset train yang seimbang untuk diagnosis, bukan estimasi performa populasi. Bobot kelas tetap dihitung dari seluruh train. Validation hanya untuk memastikan restore Teacher sama dengan hasil tersimpan. Test CSV tidak dibaca, test image tidak diekstrak, dan tidak ada tuning dari test.

## Cell 1 — Runtime dan Drive

In [ ]:
import os, sys, subprocess, importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import tensorflow as tf
minor='.'.join(tf.__version__.split('.')[:2])
try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=minor:
    if 'tf_keras' in sys.modules:raise RuntimeError('Restart runtime sebelum mengubah tf-keras')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={minor}.0'])
import tf_keras as keras
import numpy as np, pandas as pd
import json, hashlib, zipfile, gc, traceback, ast, linecache
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
from google.colab import drive, files
from IPython.display import display
gpus=tf.config.list_physical_devices('GPU')
if not gpus:raise RuntimeError('Pilih runtime GPU di Colab')
for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)
keras.mixed_precision.set_global_policy('float32')
tf.config.experimental.enable_op_determinism()
drive.mount('/content/drive')
print(tf.__version__,keras.__version__,gpus)

## Cell 2 — Konfigurasi audit (bukan protokol training final)

In [ ]:
DATASET='cbis_ddsm'  # cbis_ddsm | cmmd | inbreast | mias
SEED=42
PER_CLASS=4  # total 8 citra train; dua view: asli dan horizontal flip
DIAGNOSTIC_BATCH=2  # perkecil ke 1 jika OOM; jangan ubah batch restore Teacher
FEATURE_TEMPERATURE=2.0  # kandidat diagnostik; bukan hasil pemilihan terbaik
LAMBDA_FEATURE=1.0      # kandidat diagnostik; bukan parameter paper asli
NORMALIZATION='standardize'  # untuk langkah gradien: raw | standardize
DOWNLOAD_REPORT=True
RESEARCH=Path('/content/drive/MyDrive/Research')
SOURCE_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1'
IDS={'cbis_ddsm':'7a4bf6d1e3f6','cmmd':'f81ede1d807a','inbreast':'3f2fbdc2b913','mias':'672f572d6c33'}
REGISTRY_PATH=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'
assert DATASET in IDS and NORMALIZATION in ['raw','standardize']
assert PER_CLASS>=1 and DIAGNOSTIC_BATCH>=1 and FEATURE_TEMPERATURE>0 and LAMBDA_FEATURE>0
SOURCE=SOURCE_ROOT/DATASET/IDS[DATASET]
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT=RESEARCH/'outputs/benchmark_4datasets/b06a_feature_kd_audit_v1'/DATASET/stamp
OUT.mkdir(parents=True,exist_ok=False)
def read_json(p):return json.loads(Path(p).read_text())
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
def save(p,v):Path(p).write_text(json.dumps(v,indent=2,default=str))
CONFIG=dict(dataset=DATASET,seed=SEED,per_class=PER_CLASS,diagnostic_batch=DIAGNOSTIC_BATCH,
    temperature=FEATURE_TEMPERATURE,lambda_feature=LAMBDA_FEATURE,normalization=NORMALIZATION,
    method='SSID-inspired classification adaptation; feature KL; not original reproduction',
    feature='GAP before dropout/classifier',test_access=False,full_training=False)
save(OUT/'audit_config.json',CONFIG)
ERRORS=[]; RESTORES=[]; FEATURE_ROWS=[]; GRAD_ROWS=[]; SOURCE_HASHES={}
print('Output baru:',OUT)

## Cell 3 — Baca helper yang sudah diverifikasi dan data train/validation

In [ ]:
proto=read_json(SOURCE/'protocol.json')
if hashlib.sha256(json.dumps(proto,sort_keys=True).encode()).hexdigest()[:12]!=IDS[DATASET]:
    raise RuntimeError('ID protokol B05B tidak cocok')
helper_path=SOURCE/'b04_compat.py'
expected='2822801a4f73ab90d6df8215d12fad03ea832cbc2300bdeb36d729d66ab66297'
if sha(helper_path)!=expected or proto['b04_compat_sha256']!=expected:raise RuntimeError('Helper berubah')
# Muat hanya fungsi model/inferensi yang dibutuhkan. Tidak memanggil loader test atau training.
source_text=helper_path.read_text(); tree=ast.parse(source_text)
keep={'REGISTRY','json_safe','save_json','file_sha','metrics','make_dataset','build_model','weight_snapshot'}
nodes=[n for n in tree.body if isinstance(n,(ast.Import,ast.ImportFrom)) or
       (isinstance(n,ast.FunctionDef) and n.name in keep) or
       (isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='REGISTRY' for t in n.targets))]
helper_code=ast.unparse(ast.Module(body=nodes,type_ignores=[]))
filename='/content/b06a_inference_helpers.py'
Path(filename).write_text(helper_code)
linecache.cache[filename]=(len(helper_code),None,helper_code.splitlines(True),filename)
exec(compile(helper_code,filename,'exec'),globals())
(OUT/'inference_helpers.py').write_text(helper_code)
entry=read_json(REGISTRY_PATH)[DATASET]['full_512']
if entry['status']!='READY':raise RuntimeError('Cache belum READY')
folder=Path(entry['path']);meta=read_json(folder/'READY.json')
if meta['dataset']!=DATASET or meta['image_size']!=512:raise RuntimeError('Cache salah')
if meta['file_sha256']!=proto['data_sha256'] or meta['cohort_sha256']!=proto['cohort_sha256']:
    raise RuntimeError('Data berbeda dari B05B')
FRAMES={}
for split in ['train','validation']:
    path=folder/(split+'.csv')
    if sha(path)!=meta['file_sha256'][path.name]:raise RuntimeError('CSV berubah')
    f=pd.read_csv(path,dtype={'sample_id':str,'patient_id':str})
    if f[['sample_id','patient_id','label']].isna().any().any():raise RuntimeError('Missing ID/label')
    if f.sample_id.duplicated().any() or not f.label.isin([0,1]).all():raise RuntimeError('ID/label invalid')
    if set(f.split)!={split} or set(f.label)!={0,1}:raise RuntimeError('Split/kelas invalid')
    FRAMES[split]=f
if set(FRAMES['train'].patient_id)&set(FRAMES['validation'].patient_id):raise RuntimeError('Patient leakage')
if set(FRAMES['train'].sample_id)&set(FRAMES['validation'].sample_id):raise RuntimeError('Image leakage')
SUBSET=pd.concat([FRAMES['train'].query('label==@label').sample(n=min(PER_CLASS,int((FRAMES['train'].label==label).sum())),random_state=SEED) for label in [0,1]],ignore_index=True)
SUBSET.to_csv(OUT/'train_audit_subset.csv',index=False)
needed=pd.concat([SUBSET,FRAMES['validation']],ignore_index=True)
IMAGE_DIR=Path('/content/b06a_images')/DATASET/meta['file_sha256']['images.zip'][:16]
IMAGE_DIR.mkdir(parents=True,exist_ok=True)
# Opaque archive hash includes the complete archive; no test member is opened.
if sha(folder/'images.zip')!=meta['file_sha256']['images.zip']:raise RuntimeError('Archive berubah')
with zipfile.ZipFile(folder/'images.zip') as archive:
    for row in needed.itertuples():
        member=row.cache_member
        if Path(member).name!=member:raise RuntimeError('Unsafe cache member')
        dest=IMAGE_DIR/member
        if not dest.exists():dest.write_bytes(archive.read(member))
        with Image.open(dest) as im:a=np.asarray(im)
        if a.shape!=(512,512) or hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:
            raise RuntimeError('Pixel cache mismatch')
counts=FRAMES['train'].label.value_counts()
CLASS_WEIGHTS=tf.constant([len(FRAMES['train'])/(2*counts[i]) for i in [0,1]],tf.float32)
SOURCES={}
for teacher in ['vgg16','densenet121']:
    src=proto['teacher_sources'][teacher];p=Path(src['folder'])
    if sha(p/'config.json')!=src['config_sha256']:raise RuntimeError('Teacher config berubah')
    done=read_json(p/'COMPLETE.json')
    if done['best_checkpoint']!=src['checkpoint']:raise RuntimeError('Teacher checkpoint berubah')
    for name,digest in src['checkpoint_sha256'].items():
        part=Path(src['checkpoint']).parent/name
        if sha(part)!=digest:raise RuntimeError('Teacher weights berubah')
        SOURCE_HASHES[str(part)]=digest
    SOURCE_HASHES[str(p/'config.json')]=sha(p/'config.json')
    SOURCES[teacher]=src
save(OUT/'source_reference.json',dict(protocol=proto,teacher_sources=SOURCES,hashes=SOURCE_HASHES))
print('Train:',len(FRAMES['train']),'Validation:',len(FRAMES['validation']),'Audit train:',len(SUBSET))
print('Class weights:',CLASS_WEIGHTS.numpy(),'| Test tidak dibuka')

## Cell 4 — Definisi adaptasi dan uji loss
$L=L_{sup}+\lambda T_f^2\,\mathrm{mean}_B[\sum_c q_T(c)(\log q_T(c)-\log q_S(c))]$.

$q_T=\mathrm{softmax}(N(h_T)/T_f)$ dan $q_S=\mathrm{softmax}(N(P(h_S))/T_f)$. Teacher dihentikan gradiennya. $N$ adalah identitas (`raw`) atau standardisasi per citra lintas kanal (`standardize`, epsilon 1e-5). Ini mengubah makna representasi dan merupakan pilihan adaptasi yang harus dilaporkan. Raw dan standardized hanya dibandingkan sebagai diagnosis skala, tanpa memilih berdasarkan test.

Supervised loss adalah mean BCE berbobot kelas train. Tidak ada logit Hinton loss. Temperatur fitur tidak identik secara makna dengan temperatur probabilitas kelas.

In [ ]:
def raw_batch(frame,flip=False):
    a=np.stack([np.asarray(Image.open(IMAGE_DIR/n).convert('RGB'),dtype=np.float32) for n in frame.cache_member])
    if flip:a=a[:,:,::-1,:].copy()
    return tf.constant(a)
def prep(x,key):return getattr(keras.applications,REGISTRY[key][2]).preprocess_input(tf.identity(x))
def feature_model(model):
    # Input dropout adalah pooled backbone representation, sebelum dropout.
    return keras.Model(model.input,model.get_layer('head_dropout').input)
def normalize(h,mode):
    h=tf.cast(h,tf.float32)
    if mode=='raw':return h
    mean,var=tf.nn.moments(h,axes=[-1],keepdims=True)
    return (h-mean)*tf.math.rsqrt(var+1e-5)
def feature_loss(ht,hs,mode):
    lt=tf.nn.log_softmax(tf.stop_gradient(normalize(ht,mode))/FEATURE_TEMPERATURE,axis=-1)
    ls=tf.nn.log_softmax(normalize(hs,mode)/FEATURE_TEMPERATURE,axis=-1)
    return FEATURE_TEMPERATURE**2*tf.reduce_mean(tf.reduce_sum(tf.exp(lt)*(lt-ls),axis=-1))
def supervised(y,z):
    y=tf.cast(tf.reshape(y,[-1,1]),tf.float32)
    w=tf.gather(CLASS_WEIGHTS,tf.cast(tf.reshape(y,[-1]),tf.int32))
    return tf.reduce_mean(w*tf.reshape(tf.nn.sigmoid_cross_entropy_with_logits(labels=y,logits=z),[-1]))
def weight_hash(model):
    h=hashlib.sha256()
    for w in model.weights:h.update(w.numpy().tobytes())
    return h.hexdigest()
def grad_norm(grads):
    present=[g for g in grads if g is not None]
    for g in present:tf.debugging.assert_all_finite(g,'Nonfinite gradient')
    return float(tf.linalg.global_norm(present).numpy()) if present else 0.
# Loss harus nol pada distribusi identik; stop_gradient teacher; student menerima gradien.
a=tf.Variable([[1.,2.,-1.],[2.,0.,1.]])
b=tf.Variable([[0.,1.,2.],[1.,3.,0.]])
for mode in ['raw','standardize']:
    assert abs(float(feature_loss(a,a,mode)))<1e-5
    with tf.GradientTape(persistent=True) as tape:l=feature_loss(a,b,mode)
    assert float(l)>=-1e-6 and tape.gradient(l,a) is None
    assert grad_norm(tape.gradient(l,[b]))>0
    del tape
print('Loss unit checks passed')

## Cell 5 — Audit Teacher, empat pasangan, dan satu langkah sementara
Cell menangkap kegagalan agar ZIP tetap dapat dibuat. Restore menggunakan batch 8 seperti referensi B04. Audit semua pasangan dilakukan berurutan agar memori lebih hemat. Satu langkah gradien meniru fine-tuning; tidak mengukur performa training akhir.

In [ ]:
def audit_pair(teacher,student):
    keras.backend.clear_session();gc.collect();keras.utils.set_random_seed(SEED)
    src=SOURCES[teacher]
    tm=build_model(teacher,512,SEED,weights=None,stage='head')
    weight_snapshot(tm).restore(src['checkpoint']).assert_existing_objects_matched()
    tm.trainable=False
    teacher_before=weight_hash(tm)
    if teacher not in {r['teacher'] for r in RESTORES}:
        saved=pd.read_csv(Path(src['folder'])/'validation_predictions.csv',dtype={'sample_id':str,'patient_id':str},float_precision='round_trip')
        ref=FRAMES['validation']
        if saved.sample_id.duplicated().any() or set(saved.sample_id)!=set(ref.sample_id):raise RuntimeError('Teacher sample IDs mismatch')
        saved=saved.set_index('sample_id').loc[ref.sample_id].reset_index()
        for c in ['patient_id','label']:
            if not np.array_equal(saved[c].to_numpy(),ref[c].to_numpy()):raise RuntimeError('Teacher labels/IDs mismatch')
        pred=tm.predict(make_dataset(ref,IMAGE_DIR,teacher,8,SEED,size=512),verbose=0).reshape(-1)
        old=saved.probability.to_numpy(float)
        ok=bool(np.allclose(pred,old,atol=1e-5,rtol=1e-5) and np.array_equal(pred>=.5,old>=.5))
        RESTORES.append(dict(teacher=teacher,passed=ok,max_abs_difference=float(np.max(abs(pred-old)))))
        if not ok:raise RuntimeError('Teacher restore failed; jangan memperbesar toleransi tanpa diagnosis')
    elif not next(r['passed'] for r in RESTORES if r['teacher']==teacher):raise RuntimeError('Prior teacher restore failed')
    sm=build_model(student,512,SEED,weights='imagenet',stage='finetune')
    tfm=feature_model(tm);sfm=feature_model(sm)
    dt=int(tfm.output_shape[-1]);ds=int(sfm.output_shape[-1])
    projector=keras.layers.Dense(dt,use_bias=True,kernel_initializer=keras.initializers.GlorotUniform(seed=SEED+100),name='feature_projector')
    projector(tf.zeros([1,ds]))
    initial_student=weight_hash(sm)
    for flip in [False,True]:
        for start in range(0,len(SUBSET),DIAGNOSTIC_BATCH):
            f=SUBSET.iloc[start:start+DIAGNOSTIC_BATCH];x=raw_batch(f,flip)
            ht=tfm(prep(x,teacher),training=False);hs=sfm(prep(x,student),training=False);hp=projector(hs)
            for mode in ['raw','standardize']:
                qt=tf.nn.softmax(normalize(ht,mode)/FEATURE_TEMPERATURE,axis=-1)
                entropy=-tf.reduce_mean(tf.reduce_sum(qt*tf.math.log(tf.maximum(qt,1e-30)),axis=-1))/np.log(dt)
                FEATURE_ROWS.append(dict(teacher=teacher,student=student,flip=flip,batch_start=start,
                    teacher_dim=dt,student_dim=ds,normalization=mode,
                    teacher_std=float(tf.math.reduce_std(ht)),student_std=float(tf.math.reduce_std(hs)),
                    projected_std=float(tf.math.reduce_std(hp)),normalized_teacher_entropy=float(entropy),
                    feature_kl=float(feature_loss(ht,hp,mode))))
    # Minibatch berisi kedua kelas bila batch>=2; jika batch=1 hanya audit gradien satu citra.
    f=pd.concat([SUBSET[SUBSET.label==0].iloc[:1],SUBSET[SUBSET.label==1].iloc[:1]]).iloc[:DIAGNOSTIC_BATCH]
    x=raw_batch(f);y=tf.constant(f.label.to_numpy(),tf.float32)
    variables=sm.trainable_variables+projector.trainable_variables
    with tf.GradientTape(persistent=True) as tape:
        ht=tfm(prep(x,teacher),training=False)
        hs=sfm(prep(x,student),training=False)
        dropped=sm.get_layer('head_dropout')(hs,training=True)
        classifier=sm.get_layer('probability')
        logits=tf.matmul(dropped,classifier.kernel)+classifier.bias
        ls=supervised(y,logits);lk=feature_loss(ht,projector(hs),NORMALIZATION)
        total=ls+LAMBDA_FEATURE*lk
    gs=tape.gradient(ls,sm.trainable_variables);gk=tape.gradient(lk,sm.trainable_variables)
    grads=tape.gradient(total,variables);gt=tape.gradient(total,tm.weights)
    gp=tape.gradient(lk,projector.trainable_variables)
    base=next(l for l in sm.layers if isinstance(l,keras.Model))
    gb=tape.gradient(lk,base.trainable_variables)
    ns,nk,np_,nb=grad_norm(gs),grad_norm(gk),grad_norm(gp),grad_norm(gb)
    if any(g is not None for g in gt):raise RuntimeError('Teacher received gradients')
    if min(ns,nk,np_,nb)<=0:raise RuntimeError('Missing/nonzero gradient failure')
    tf.debugging.assert_all_finite(total,'Invalid total loss');grad_norm(grads)
    projector_before=weight_hash(projector)
    optimizer=keras.optimizers.Adam(1e-5)
    optimizer.apply_gradients([(g,v) for g,v in zip(grads,variables) if g is not None])
    frozen=(teacher_before==weight_hash(tm));changed=(initial_student!=weight_hash(sm));pchanged=projector_before!=weight_hash(projector)
    # La même classifier probability tanpa projector harus identik dengan sm(training=False).
    feat=sfm(prep(x,student),training=False)
    direct=tf.sigmoid(tf.matmul(feat,classifier.kernel)+classifier.bias)
    native=sm(prep(x,student),training=False)
    diff=float(tf.reduce_max(abs(direct-native)))
    passed=bool(frozen and changed and pchanged and diff<=1e-6)
    GRAD_ROWS.append(dict(teacher=teacher,student=student,passed=passed,supervised_loss=float(ls),feature_loss=float(lk),
        weighted_feature_to_supervised_loss=float(LAMBDA_FEATURE*lk/(ls+1e-12)),supervised_grad_norm=ns,feature_grad_norm=nk,
        weighted_feature_to_supervised_grad=LAMBDA_FEATURE*nk/(ns+1e-12),projector_grad_norm=np_,backbone_feature_grad_norm=nb,
        teacher_unchanged=frozen,student_changed=changed,projector_changed=pchanged,inference_without_projector_maxdiff=diff,
        initial_student_sha256=initial_student,projector_parameters=projector.count_params()))
    if not passed:raise RuntimeError('One-step invariants failed')
    del tape,optimizer,tm,sm,tfm,sfm,projector
for teacher in ['vgg16','densenet121']:
    for student in ['mobilenet_v2','nasnet_mobile']:
        print('AUDIT',teacher,'->',student,flush=True)
        try:audit_pair(teacher,student)
        except Exception as exc:
            ERRORS.append(dict(teacher=teacher,student=student,error=str(exc),traceback=traceback.format_exc()))
            print('FAILED:',exc)
        finally:
            keras.backend.clear_session();gc.collect()
            pd.DataFrame(RESTORES).to_csv(OUT/'teacher_restore_audit.csv',index=False)
            pd.DataFrame(FEATURE_ROWS).to_csv(OUT/'feature_distribution_audit.csv',index=False)
            pd.DataFrame(GRAD_ROWS).to_csv(OUT/'gradient_update_audit.csv',index=False)
            save(OUT/'errors.json',ERRORS)
print('Lanjutkan Cell 6, termasuk bila ada kegagalan.')

## Cell 6 — Ringkasan dan ZIP
PASS hanya berarti mekanisme teknis lolos, bukan bukti adaptasi efektif atau hiperparameter optimal. Rasio loss/gradien dan entropy diperiksa sebelum B06B. Gradient ratio bersifat diagnostik: arah gradien juga memengaruhi optimisasi. B06B belum dijalankan oleh notebook ini.

In [ ]:
(OUT/'b06a_audit_source.py').write_text("import os, sys, subprocess, importlib.metadata as metadata\nos.environ['TF_CPP_MIN_LOG_LEVEL']='2'\nimport tensorflow as tf\nminor='.'.join(tf.__version__.split('.')[:2])\ntry: installed=metadata.version('tf-keras')\nexcept metadata.PackageNotFoundError: installed=''\nif '.'.join(installed.split('.')[:2])!=minor:\n    if 'tf_keras' in sys.modules:raise RuntimeError('Restart runtime sebelum mengubah tf-keras')\n    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={minor}.0'])\nimport tf_keras as keras\nimport numpy as np, pandas as pd\nimport json, hashlib, zipfile, gc, traceback, ast, linecache\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom PIL import Image\nfrom google.colab import drive, files\nfrom IPython.display import display\ngpus=tf.config.list_physical_devices('GPU')\nif not gpus:raise RuntimeError('Pilih runtime GPU di Colab')\nfor gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)\nkeras.mixed_precision.set_global_policy('float32')\ntf.config.experimental.enable_op_determinism()\ndrive.mount('/content/drive')\nprint(tf.__version__,keras.__version__,gpus)\n\nDATASET='cbis_ddsm'  # cbis_ddsm | cmmd | inbreast | mias\nSEED=42\nPER_CLASS=4  # total 8 citra train; dua view: asli dan horizontal flip\nDIAGNOSTIC_BATCH=2  # perkecil ke 1 jika OOM; jangan ubah batch restore Teacher\nFEATURE_TEMPERATURE=2.0  # kandidat diagnostik; bukan hasil pemilihan terbaik\nLAMBDA_FEATURE=1.0      # kandidat diagnostik; bukan parameter paper asli\nNORMALIZATION='standardize'  # untuk langkah gradien: raw | standardize\nDOWNLOAD_REPORT=True\nRESEARCH=Path('/content/drive/MyDrive/Research')\nSOURCE_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1'\nIDS={'cbis_ddsm':'7a4bf6d1e3f6','cmmd':'f81ede1d807a','inbreast':'3f2fbdc2b913','mias':'672f572d6c33'}\nREGISTRY_PATH=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'\nassert DATASET in IDS and NORMALIZATION in ['raw','standardize']\nassert PER_CLASS>=1 and DIAGNOSTIC_BATCH>=1 and FEATURE_TEMPERATURE>0 and LAMBDA_FEATURE>0\nSOURCE=SOURCE_ROOT/DATASET/IDS[DATASET]\nstamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')\nOUT=RESEARCH/'outputs/benchmark_4datasets/b06a_feature_kd_audit_v1'/DATASET/stamp\nOUT.mkdir(parents=True,exist_ok=False)\ndef read_json(p):return json.loads(Path(p).read_text())\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,'rb') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)\n    return h.hexdigest()\ndef save(p,v):Path(p).write_text(json.dumps(v,indent=2,default=str))\nCONFIG=dict(dataset=DATASET,seed=SEED,per_class=PER_CLASS,diagnostic_batch=DIAGNOSTIC_BATCH,\n    temperature=FEATURE_TEMPERATURE,lambda_feature=LAMBDA_FEATURE,normalization=NORMALIZATION,\n    method='SSID-inspired classification adaptation; feature KL; not original reproduction',\n    feature='GAP before dropout/classifier',test_access=False,full_training=False)\nsave(OUT/'audit_config.json',CONFIG)\nERRORS=[]; RESTORES=[]; FEATURE_ROWS=[]; GRAD_ROWS=[]; SOURCE_HASHES={}\nprint('Output baru:',OUT)\n\nproto=read_json(SOURCE/'protocol.json')\nif hashlib.sha256(json.dumps(proto,sort_keys=True).encode()).hexdigest()[:12]!=IDS[DATASET]:\n    raise RuntimeError('ID protokol B05B tidak cocok')\nhelper_path=SOURCE/'b04_compat.py'\nexpected='2822801a4f73ab90d6df8215d12fad03ea832cbc2300bdeb36d729d66ab66297'\nif sha(helper_path)!=expected or proto['b04_compat_sha256']!=expected:raise RuntimeError('Helper berubah')\n# Muat hanya fungsi model/inferensi yang dibutuhkan. Tidak memanggil loader test atau training.\nsource_text=helper_path.read_text(); tree=ast.parse(source_text)\nkeep={'REGISTRY','json_safe','save_json','file_sha','metrics','make_dataset','build_model','weight_snapshot'}\nnodes=[n for n in tree.body if isinstance(n,(ast.Import,ast.ImportFrom)) or\n       (isinstance(n,ast.FunctionDef) and n.name in keep) or\n       (isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='REGISTRY' for t in n.targets))]\nhelper_code=ast.unparse(ast.Module(body=nodes,type_ignores=[]))\nfilename='/content/b06a_inference_helpers.py'\nPath(filename).write_text(helper_code)\nlinecache.cache[filename]=(len(helper_code),None,helper_code.splitlines(True),filename)\nexec(compile(helper_code,filename,'exec'),globals())\n(OUT/'inference_helpers.py').write_text(helper_code)\nentry=read_json(REGISTRY_PATH)[DATASET]['full_512']\nif entry['status']!='READY':raise RuntimeError('Cache belum READY')\nfolder=Path(entry['path']);meta=read_json(folder/'READY.json')\nif meta['dataset']!=DATASET or meta['image_size']!=512:raise RuntimeError('Cache salah')\nif meta['file_sha256']!=proto['data_sha256'] or meta['cohort_sha256']!=proto['cohort_sha256']:\n    raise RuntimeError('Data berbeda dari B05B')\nFRAMES={}\nfor split in ['train','validation']:\n    path=folder/(split+'.csv')\n    if sha(path)!=meta['file_sha256'][path.name]:raise RuntimeError('CSV berubah')\n    f=pd.read_csv(path,dtype={'sample_id':str,'patient_id':str})\n    if f[['sample_id','patient_id','label']].isna().any().any():raise RuntimeError('Missing ID/label')\n    if f.sample_id.duplicated().any() or not f.label.isin([0,1]).all():raise RuntimeError('ID/label invalid')\n    if set(f.split)!={split} or set(f.label)!={0,1}:raise RuntimeError('Split/kelas invalid')\n    FRAMES[split]=f\nif set(FRAMES['train'].patient_id)&set(FRAMES['validation'].patient_id):raise RuntimeError('Patient leakage')\nif set(FRAMES['train'].sample_id)&set(FRAMES['validation'].sample_id):raise RuntimeError('Image leakage')\nSUBSET=pd.concat([FRAMES['train'].query('label==@label').sample(n=min(PER_CLASS,int((FRAMES['train'].label==label).sum())),random_state=SEED) for label in [0,1]],ignore_index=True)\nSUBSET.to_csv(OUT/'train_audit_subset.csv',index=False)\nneeded=pd.concat([SUBSET,FRAMES['validation']],ignore_index=True)\nIMAGE_DIR=Path('/content/b06a_images')/DATASET/meta['file_sha256']['images.zip'][:16]\nIMAGE_DIR.mkdir(parents=True,exist_ok=True)\n# Opaque archive hash includes the complete archive; no test member is opened.\nif sha(folder/'images.zip')!=meta['file_sha256']['images.zip']:raise RuntimeError('Archive berubah')\nwith zipfile.ZipFile(folder/'images.zip') as archive:\n    for row in needed.itertuples():\n        member=row.cache_member\n        if Path(member).name!=member:raise RuntimeError('Unsafe cache member')\n        dest=IMAGE_DIR/member\n        if not dest.exists():dest.write_bytes(archive.read(member))\n        with Image.open(dest) as im:a=np.asarray(im)\n        if a.shape!=(512,512) or hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:\n            raise RuntimeError('Pixel cache mismatch')\ncounts=FRAMES['train'].label.value_counts()\nCLASS_WEIGHTS=tf.constant([len(FRAMES['train'])/(2*counts[i]) for i in [0,1]],tf.float32)\nSOURCES={}\nfor teacher in ['vgg16','densenet121']:\n    src=proto['teacher_sources'][teacher];p=Path(src['folder'])\n    if sha(p/'config.json')!=src['config_sha256']:raise RuntimeError('Teacher config berubah')\n    done=read_json(p/'COMPLETE.json')\n    if done['best_checkpoint']!=src['checkpoint']:raise RuntimeError('Teacher checkpoint berubah')\n    for name,digest in src['checkpoint_sha256'].items():\n        part=Path(src['checkpoint']).parent/name\n        if sha(part)!=digest:raise RuntimeError('Teacher weights berubah')\n        SOURCE_HASHES[str(part)]=digest\n    SOURCE_HASHES[str(p/'config.json')]=sha(p/'config.json')\n    SOURCES[teacher]=src\nsave(OUT/'source_reference.json',dict(protocol=proto,teacher_sources=SOURCES,hashes=SOURCE_HASHES))\nprint('Train:',len(FRAMES['train']),'Validation:',len(FRAMES['validation']),'Audit train:',len(SUBSET))\nprint('Class weights:',CLASS_WEIGHTS.numpy(),'| Test tidak dibuka')\n\ndef raw_batch(frame,flip=False):\n    a=np.stack([np.asarray(Image.open(IMAGE_DIR/n).convert('RGB'),dtype=np.float32) for n in frame.cache_member])\n    if flip:a=a[:,:,::-1,:].copy()\n    return tf.constant(a)\ndef prep(x,key):return getattr(keras.applications,REGISTRY[key][2]).preprocess_input(tf.identity(x))\ndef feature_model(model):\n    # Input dropout adalah pooled backbone representation, sebelum dropout.\n    return keras.Model(model.input,model.get_layer('head_dropout').input)\ndef normalize(h,mode):\n    h=tf.cast(h,tf.float32)\n    if mode=='raw':return h\n    mean,var=tf.nn.moments(h,axes=[-1],keepdims=True)\n    return (h-mean)*tf.math.rsqrt(var+1e-5)\ndef feature_loss(ht,hs,mode):\n    lt=tf.nn.log_softmax(tf.stop_gradient(normalize(ht,mode))/FEATURE_TEMPERATURE,axis=-1)\n    ls=tf.nn.log_softmax(normalize(hs,mode)/FEATURE_TEMPERATURE,axis=-1)\n    return FEATURE_TEMPERATURE**2*tf.reduce_mean(tf.reduce_sum(tf.exp(lt)*(lt-ls),axis=-1))\ndef supervised(y,z):\n    y=tf.cast(tf.reshape(y,[-1,1]),tf.float32)\n    w=tf.gather(CLASS_WEIGHTS,tf.cast(tf.reshape(y,[-1]),tf.int32))\n    return tf.reduce_mean(w*tf.reshape(tf.nn.sigmoid_cross_entropy_with_logits(labels=y,logits=z),[-1]))\ndef weight_hash(model):\n    h=hashlib.sha256()\n    for w in model.weights:h.update(w.numpy().tobytes())\n    return h.hexdigest()\ndef grad_norm(grads):\n    present=[g for g in grads if g is not None]\n    for g in present:tf.debugging.assert_all_finite(g,'Nonfinite gradient')\n    return float(tf.linalg.global_norm(present).numpy()) if present else 0.\n# Loss harus nol pada distribusi identik; stop_gradient teacher; student menerima gradien.\na=tf.Variable([[1.,2.,-1.],[2.,0.,1.]])\nb=tf.Variable([[0.,1.,2.],[1.,3.,0.]])\nfor mode in ['raw','standardize']:\n    assert abs(float(feature_loss(a,a,mode)))<1e-5\n    with tf.GradientTape(persistent=True) as tape:l=feature_loss(a,b,mode)\n    assert float(l)>=-1e-6 and tape.gradient(l,a) is None\n    assert grad_norm(tape.gradient(l,[b]))>0\n    del tape\nprint('Loss unit checks passed')\n\ndef audit_pair(teacher,student):\n    keras.backend.clear_session();gc.collect();keras.utils.set_random_seed(SEED)\n    src=SOURCES[teacher]\n    tm=build_model(teacher,512,SEED,weights=None,stage='head')\n    weight_snapshot(tm).restore(src['checkpoint']).assert_existing_objects_matched()\n    tm.trainable=False\n    teacher_before=weight_hash(tm)\n    if teacher not in {r['teacher'] for r in RESTORES}:\n        saved=pd.read_csv(Path(src['folder'])/'validation_predictions.csv',dtype={'sample_id':str,'patient_id':str},float_precision='round_trip')\n        ref=FRAMES['validation']\n        if saved.sample_id.duplicated().any() or set(saved.sample_id)!=set(ref.sample_id):raise RuntimeError('Teacher sample IDs mismatch')\n        saved=saved.set_index('sample_id').loc[ref.sample_id].reset_index()\n        for c in ['patient_id','label']:\n            if not np.array_equal(saved[c].to_numpy(),ref[c].to_numpy()):raise RuntimeError('Teacher labels/IDs mismatch')\n        pred=tm.predict(make_dataset(ref,IMAGE_DIR,teacher,8,SEED,size=512),verbose=0).reshape(-1)\n        old=saved.probability.to_numpy(float)\n        ok=bool(np.allclose(pred,old,atol=1e-5,rtol=1e-5) and np.array_equal(pred>=.5,old>=.5))\n        RESTORES.append(dict(teacher=teacher,passed=ok,max_abs_difference=float(np.max(abs(pred-old)))))\n        if not ok:raise RuntimeError('Teacher restore failed; jangan memperbesar toleransi tanpa diagnosis')\n    elif not next(r['passed'] for r in RESTORES if r['teacher']==teacher):raise RuntimeError('Prior teacher restore failed')\n    sm=build_model(student,512,SEED,weights='imagenet',stage='finetune')\n    tfm=feature_model(tm);sfm=feature_model(sm)\n    dt=int(tfm.output_shape[-1]);ds=int(sfm.output_shape[-1])\n    projector=keras.layers.Dense(dt,use_bias=True,kernel_initializer=keras.initializers.GlorotUniform(seed=SEED+100),name='feature_projector')\n    projector(tf.zeros([1,ds]))\n    initial_student=weight_hash(sm)\n    for flip in [False,True]:\n        for start in range(0,len(SUBSET),DIAGNOSTIC_BATCH):\n            f=SUBSET.iloc[start:start+DIAGNOSTIC_BATCH];x=raw_batch(f,flip)\n            ht=tfm(prep(x,teacher),training=False);hs=sfm(prep(x,student),training=False);hp=projector(hs)\n            for mode in ['raw','standardize']:\n                qt=tf.nn.softmax(normalize(ht,mode)/FEATURE_TEMPERATURE,axis=-1)\n                entropy=-tf.reduce_mean(tf.reduce_sum(qt*tf.math.log(tf.maximum(qt,1e-30)),axis=-1))/np.log(dt)\n                FEATURE_ROWS.append(dict(teacher=teacher,student=student,flip=flip,batch_start=start,\n                    teacher_dim=dt,student_dim=ds,normalization=mode,\n                    teacher_std=float(tf.math.reduce_std(ht)),student_std=float(tf.math.reduce_std(hs)),\n                    projected_std=float(tf.math.reduce_std(hp)),normalized_teacher_entropy=float(entropy),\n                    feature_kl=float(feature_loss(ht,hp,mode))))\n    # Minibatch berisi kedua kelas bila batch>=2; jika batch=1 hanya audit gradien satu citra.\n    f=pd.concat([SUBSET[SUBSET.label==0].iloc[:1],SUBSET[SUBSET.label==1].iloc[:1]]).iloc[:DIAGNOSTIC_BATCH]\n    x=raw_batch(f);y=tf.constant(f.label.to_numpy(),tf.float32)\n    variables=sm.trainable_variables+projector.trainable_variables\n    with tf.GradientTape(persistent=True) as tape:\n        ht=tfm(prep(x,teacher),training=False)\n        hs=sfm(prep(x,student),training=False)\n        dropped=sm.get_layer('head_dropout')(hs,training=True)\n        classifier=sm.get_layer('probability')\n        logits=tf.matmul(dropped,classifier.kernel)+classifier.bias\n        ls=supervised(y,logits);lk=feature_loss(ht,projector(hs),NORMALIZATION)\n        total=ls+LAMBDA_FEATURE*lk\n    gs=tape.gradient(ls,sm.trainable_variables);gk=tape.gradient(lk,sm.trainable_variables)\n    grads=tape.gradient(total,variables);gt=tape.gradient(total,tm.weights)\n    gp=tape.gradient(lk,projector.trainable_variables)\n    base=next(l for l in sm.layers if isinstance(l,keras.Model))\n    gb=tape.gradient(lk,base.trainable_variables)\n    ns,nk,np_,nb=grad_norm(gs),grad_norm(gk),grad_norm(gp),grad_norm(gb)\n    if any(g is not None for g in gt):raise RuntimeError('Teacher received gradients')\n    if min(ns,nk,np_,nb)<=0:raise RuntimeError('Missing/nonzero gradient failure')\n    tf.debugging.assert_all_finite(total,'Invalid total loss');grad_norm(grads)\n    projector_before=weight_hash(projector)\n    optimizer=keras.optimizers.Adam(1e-5)\n    optimizer.apply_gradients([(g,v) for g,v in zip(grads,variables) if g is not None])\n    frozen=(teacher_before==weight_hash(tm));changed=(initial_student!=weight_hash(sm));pchanged=projector_before!=weight_hash(projector)\n    # La même classifier probability tanpa projector harus identik dengan sm(training=False).\n    feat=sfm(prep(x,student),training=False)\n    direct=tf.sigmoid(tf.matmul(feat,classifier.kernel)+classifier.bias)\n    native=sm(prep(x,student),training=False)\n    diff=float(tf.reduce_max(abs(direct-native)))\n    passed=bool(frozen and changed and pchanged and diff<=1e-6)\n    GRAD_ROWS.append(dict(teacher=teacher,student=student,passed=passed,supervised_loss=float(ls),feature_loss=float(lk),\n        weighted_feature_to_supervised_loss=float(LAMBDA_FEATURE*lk/(ls+1e-12)),supervised_grad_norm=ns,feature_grad_norm=nk,\n        weighted_feature_to_supervised_grad=LAMBDA_FEATURE*nk/(ns+1e-12),projector_grad_norm=np_,backbone_feature_grad_norm=nb,\n        teacher_unchanged=frozen,student_changed=changed,projector_changed=pchanged,inference_without_projector_maxdiff=diff,\n        initial_student_sha256=initial_student,projector_parameters=projector.count_params()))\n    if not passed:raise RuntimeError('One-step invariants failed')\n    del tape,optimizer,tm,sm,tfm,sfm,projector\nfor teacher in ['vgg16','densenet121']:\n    for student in ['mobilenet_v2','nasnet_mobile']:\n        print('AUDIT',teacher,'->',student,flush=True)\n        try:audit_pair(teacher,student)\n        except Exception as exc:\n            ERRORS.append(dict(teacher=teacher,student=student,error=str(exc),traceback=traceback.format_exc()))\n            print('FAILED:',exc)\n        finally:\n            keras.backend.clear_session();gc.collect()\n            pd.DataFrame(RESTORES).to_csv(OUT/'teacher_restore_audit.csv',index=False)\n            pd.DataFrame(FEATURE_ROWS).to_csv(OUT/'feature_distribution_audit.csv',index=False)\n            pd.DataFrame(GRAD_ROWS).to_csv(OUT/'gradient_update_audit.csv',index=False)\n            save(OUT/'errors.json',ERRORS)\nprint('Lanjutkan Cell 6, termasuk bila ada kegagalan.')\n\nfor path,digest in SOURCE_HASHES.items():\n    if sha(path)!=digest:ERRORS.append(dict(error='Source changed during audit',path=path))\ngrad=pd.DataFrame(GRAD_ROWS)\nmatched=True\nif len(grad):\n    matched=bool((grad.groupby('student').initial_student_sha256.nunique()==1).all())\n    display(grad.round(6))\nif not matched:ERRORS.append(dict(error='Student initializations differ across teachers'))\npassed=bool(len(GRAD_ROWS)==4 and len(RESTORES)==2 and all(r['passed'] for r in GRAD_ROWS+RESTORES) and not ERRORS and matched)\nfeature=pd.DataFrame(FEATURE_ROWS)\nif len(feature):display(feature.groupby(['teacher','student','normalization'])[['feature_kl','normalized_teacher_entropy']].mean().round(6))\nsave(OUT/'errors.json',ERRORS)\nsave(OUT/'SUMMARY.json',dict(technical_audit_passed=passed,completed_pairs=len(GRAD_ROWS),\n    matched_student_initialization=matched,test_evaluated=False,full_training=False,\n    next_step='Review loss/gradient scale before defining B06B; PASS does not establish efficacy'))\nsave(OUT/'environment.json',dict(tensorflow=tf.__version__,tf_keras=keras.__version__,gpu=[tf.config.experimental.get_device_details(g) for g in gpus]))\nREPORT=OUT/f'B06A_{DATASET}_{stamp}_report.zip'\nwith zipfile.ZipFile(REPORT,'w',zipfile.ZIP_DEFLATED) as z:\n    for p in sorted(OUT.iterdir()):\n        if p.is_file() and p.suffix in ['.json','.csv','.py']:z.write(p,p.name)\nprint('TECHNICAL AUDIT:', 'PASS' if passed else 'FAIL — lihat errors.json')\nprint(REPORT)\nif DOWNLOAD_REPORT:files.download(str(REPORT))",encoding='utf-8')
for path,digest in SOURCE_HASHES.items():
    if sha(path)!=digest:ERRORS.append(dict(error='Source changed during audit',path=path))
grad=pd.DataFrame(GRAD_ROWS)
matched=True
if len(grad):
    matched=bool((grad.groupby('student').initial_student_sha256.nunique()==1).all())
    display(grad.round(6))
if not matched:ERRORS.append(dict(error='Student initializations differ across teachers'))
passed=bool(len(GRAD_ROWS)==4 and len(RESTORES)==2 and all(r['passed'] for r in GRAD_ROWS+RESTORES) and not ERRORS and matched)
feature=pd.DataFrame(FEATURE_ROWS)
if len(feature):display(feature.groupby(['teacher','student','normalization'])[['feature_kl','normalized_teacher_entropy']].mean().round(6))
save(OUT/'errors.json',ERRORS)
save(OUT/'SUMMARY.json',dict(technical_audit_passed=passed,completed_pairs=len(GRAD_ROWS),
    matched_student_initialization=matched,test_evaluated=False,full_training=False,
    next_step='Review loss/gradient scale before defining B06B; PASS does not establish efficacy'))
save(OUT/'environment.json',dict(tensorflow=tf.__version__,tf_keras=keras.__version__,gpu=[tf.config.experimental.get_device_details(g) for g in gpus]))
REPORT=OUT/f'B06A_{DATASET}_{stamp}_report.zip'
with zipfile.ZipFile(REPORT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p.suffix in ['.json','.csv','.py']:z.write(p,p.name)
print('TECHNICAL AUDIT:', 'PASS' if passed else 'FAIL — lihat errors.json')
print(REPORT)
if DOWNLOAD_REPORT:files.download(str(REPORT))